# Binary Classification: Non-Diabetic vs Diabetic

**Input:** `data/processed/CDC_Diabetes_Dataset_clean.csv`

**Purpose:** Build and evaluate a binary classifier (class 0 = no diabetes, class 1 = pre-diabetes or diabetes), prioritising recall for the diabetic class. Includes threshold tuning (MCC-optimised), probability calibration, permutation importance, and full SHAP + LIME local explainability across 8 case types (TP/FP/TN/FN × high-confidence and borderline).

**Pipeline:** LR baseline → class-weighted LR + threshold tuning → feature engineering (Feature Set A) → XGBoost baseline → XGBoost + Optuna tuning + threshold → isotonic calibration → permutation importance → global SHAP → local SHAP → local LIME → side-by-side SHAP/LIME comparison.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "CDC_Diabetes_Dataset_clean.csv"
FIG_DIR = PROJECT_ROOT / "figures" / "results_binary"
FIG_DIR.mkdir(parents=True, exist_ok=True)


print("Project root directory:", PROJECT_ROOT)
print("Data path exists:", DATA_PATH.exists())
assert DATA_PATH.exists(), f"Data file not found at {DATA_PATH}"
print("Figures directory:", FIG_DIR)

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass
from typing import Dict, Any, List

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, 
    classification_report, confusion_matrix, log_loss, precision_recall_fscore_support, recall_score, 
    brier_score_loss, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, auc)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.base import clone
from sklearn.metrics import matthews_corrcoef


from xgboost import XGBClassifier
import optuna

import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn import set_config
set_config(display="diagram")
#lr_raw_none_default

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# -------------------------------
# Experiment management utilities
# -------------------------------



# Each ExperimentResult object stores the outcome of one model run
# This is purely for organisation and comparison and not modelling

@dataclass
class ExperimentResult:
    exp_id: str                 # Unique experiment name (used in report)
    model_family: str           # e.g. "LR", "XGB"
    features: str               # e.g. "raw", "feateng"
    sampling: str               # "none", "classweight", "smote"
    tuning: str                 # "default", "grid", "optuna"
    notes: str                  # Short description
    metrics: Dict[str, float]   # Evaluation metrics (F1, log loss, etc.)
    params: Dict[str, Any]      # Model hyperparameters


# List that will store ALL experiments run in this notebook
results: List[ExperimentResult] = []


def log_experiment(
    exp_id: str,
    model_family: str,
    features: str,
    sampling: str,
    tuning: str,
    notes: str,
    metrics: Dict[str, float],
    params: Dict[str, Any]
):
    """
    Store the results of a single experiment in a structured way.
    This function is called once per model run.
    """
    results.append(
        ExperimentResult(
            exp_id=exp_id,
            model_family=model_family,
            features=features,
            sampling=sampling,
            tuning=tuning,
            notes=notes,
            metrics=metrics,
            params=params
        )
    )


def results_df() -> pd.DataFrame:
    """
    Convert all logged experiments into a pandas DataFrame
    for easy comparison and reporting.
    """
    if not results:
        return pd.DataFrame()

    rows = []
    for r in results:
        row = {
            "experiment": r.exp_id,
            "model": r.model_family,
            "features": r.features,
            "sampling": r.sampling,
            "tuning": r.tuning,
            "notes": r.notes,
        }

        # Add metrics with a prefix to avoid name clashes
        for k, v in r.metrics.items():
            row[f"metric__{k}"] = v

        rows.append(row)

    return pd.DataFrame(rows)


print("Step 0 complete: experiment logger initialised.")

In [ ]:
# ---------------------------------------
# Setup: imports, paths, utilities
# ---------------------------------------

def savefig(name: str):
    """
    Save the current matplotlib figure to the figures directory
    at publication-quality resolution.
    """
    plt.savefig(FIG_DIR / name, dpi=300, bbox_inches="tight")
    print(f"Figure saved: {FIG_DIR / name}")

In [ ]:
# load data from csv
data = pd.read_csv(DATA_PATH)
print("Data loaded successfully.")
print("Top 5 rows:")
data.head()



**Baseline Binary Logistic Regression**

In [ ]:
# =======================================
# Raw feature set - baseline (BINARY)
# =======================================

# Create binary target: 0 = Non-Diabetic, 1 = Diabetic (prediabetes + diabetes combined)
data['Diabetes_binary'] = (data['Diabetes_012'] > 0).astype(int)

print("Original 3-way distribution:")
print(data['Diabetes_012'].value_counts().sort_index())
print("\nBinary target distribution:")
print(data['Diabetes_binary'].value_counts().sort_index())
print("\nBinary class proportions (%):")
print((data['Diabetes_binary'].value_counts(normalize=True).sort_index() * 100).round(2))

target = "Diabetes_binary"

binary_features = [
    "HighBP","HighChol","CholCheck","Smoker","Stroke","HeartDiseaseorAttack",
    "PhysActivity","Fruits","Veggies","HvyAlcoholConsump","AnyHealthcare",
    "NoDocbcCost","DiffWalk","Sex"]

ordinal_features = ["GenHlth","Age","Education","Income"]
numeric_features  = ["BMI","MentHlth","PhysHlth"]

feature_cols = binary_features + ordinal_features + numeric_features

X = data[feature_cols].copy()
y = data[target].astype(int)
print("\nX shape", X.shape)
print("Class balance(%):")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))



In [ ]:
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

X_eval_raw, X_test_raw, y_eval, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Train:", X_train_raw.shape, "\nEval:", X_eval_raw.shape, "\nTest:", X_test_raw.shape)

print("\nClass % (train):")
print((y_train.value_counts(normalize=True).sort_index()*100).round(2))
print("\nClass % (eval):")
print((y_eval.value_counts(normalize=True).sort_index()*100).round(2))
print("\nClass % (test):")
print((y_test.value_counts(normalize=True).sort_index()*100).round(2))


In [ ]:
def evaluate_binary_classifier(y_true, y_pred, y_proba):
    """
    Comprehensive evaluation metrics for binary classification.
    
    Target classes:
        0 = Non-Diabetic (healthy)
        1 = Diabetic (prediabetes + diabetes combined)
    
    Includes per-class metrics, macro/weighted averages, ROC-AUC, Brier score,
    and clinical interpretation metrics (sensitivity, specificity, PPV, NPV).
    
    CRISP-DM Methodology: Evaluation phase metrics for model assessment.
    """
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score, f1_score, precision_score,
        recall_score, precision_recall_fscore_support, log_loss, 
        brier_score_loss, roc_auc_score, matthews_corrcoef, average_precision_score
    )
    
    # Extract probability for positive class (class 1 = Diabetic)
    if y_proba.ndim == 2:
        y_proba_pos = y_proba[:, 1]
    else:
        y_proba_pos = y_proba
    
    # Per-class precision, recall, f1, support
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )

    metrics = {
        # ===== Overall metrics =====
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        
        # ===== F1 scores =====
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_class_1 (diabetic)": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        
        # ===== Probabilistic metrics =====
        "log_loss": log_loss(y_true, y_proba),
        "brier_score": brier_score_loss(y_true, y_proba_pos),
        
        # ===== ROC-AUC (binary) =====
        "roc_auc": roc_auc_score(y_true, y_proba_pos),
        
        # ===== Precision-Recall AUC (important for imbalanced data) =====
        "pr_auc": average_precision_score(y_true, y_proba_pos),
        
        # ===== Matthews Correlation Coefficient (robust for imbalance) =====
        "mcc": matthews_corrcoef(y_true, y_pred),
        
        # ===== Per-class metrics with clinical aliases =====
        # Class 0 = Non-Diabetic
        "precision_class_0 (NPV)": precision[0],
        "recall_class_0 (specificity)": recall[0],
        "f1_class_0": f1[0],
        "support_class_0": int(support[0]),
        
        # Class 1 = Diabetic
        "precision_class_1 (PPV)": precision[1],
        "recall_class_1 (sensitivity)": recall[1],
        "f1_class_1": f1[1],
        "support_class_1": int(support[1]),
    }

    return metrics

In [ ]:
# Preprocess: scale ordinal + numeric, keep binary as-is
preprocess_raw = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("ord", StandardScaler(), ordinal_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop"
)

lr_raw_none_default = Pipeline(steps=[
    ("preprocess", preprocess_raw),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
        class_weight=None  # baseline, class is not balanced
    ))
])
# sklearn handles binary classification automatically with lbfgs solver

print("Logistic regression baseline pipeline built ✅")

In [ ]:
# Fit on train set
lr_raw_none_default.fit(X_train_raw, y_train)

# Predict on eval set
y_pred_base = lr_raw_none_default.predict(X_eval_raw)
y_proba_base = lr_raw_none_default.predict_proba(X_eval_raw)

# Evaluate on eval set
metrics_lr_base = evaluate_binary_classifier(y_eval, y_pred_base, y_proba_base)
print(metrics_lr_base)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_base))
print("\nClassification report:\n", classification_report(y_eval, y_pred_base, digits=3))

# Log (baseline)
log_experiment(
    exp_id="LR__raw__none__default",
    model_family="LR",
    features="raw",
    sampling="none",
    tuning="default",
    notes="Baseline LR on raw features (no class weighting), evaluated on validation set",
    metrics=metrics_lr_base,
    params=lr_raw_none_default.get_params()
)

results_df()

# ==========================
# Report-ready confusion matrix (counts + row %), LR baseline (binary)
# ==========================
class_names = ["Non-Diabetic", "Diabetic/Prediabetic"]

# Confusion matrix (counts)
cm_lr_base = confusion_matrix(y_eval, y_pred_base, labels=[0, 1])
cm_lr_base_df = pd.DataFrame(cm_lr_base, index=class_names, columns=class_names)

# Row-normalized percentages
cm_lr_base_row_pct = cm_lr_base_df.div(cm_lr_base_df.sum(axis=1), axis=0) * 100

# Combine counts + percentages for annotation
cm_lr_base_annot = cm_lr_base_df.astype(int).astype(str) + "\n(" + cm_lr_base_row_pct.round(1).astype(str) + "%)"

# Display table (counts + row %) for direct copy into report
cm_lr_base_table = cm_lr_base_df.copy()
for r in cm_lr_base_table.index:
    for c in cm_lr_base_table.columns:
        cm_lr_base_table.loc[r, c] = f"{cm_lr_base_df.loc[r, c]} ({cm_lr_base_row_pct.loc[r, c]:.1f}%)"

print("Confusion matrix (counts and row %):")
display(cm_lr_base_table)

# Plot heatmap for report-ready figure
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_lr_base_df,
    annot=cm_lr_base_annot,
    fmt="",
    cmap="Blues",
    cbar=False,
    linewidths=0.5,
    linecolor="white"
 )
plt.title("Confusion Matrix — LR Baseline (Eval Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
savefig("confusion_matrix_lr_baseline_eval.png")
plt.show()

In [ ]:
# ==========================
# 2nd Model: Balanced LR
# LR__raw__classweight__default
# ==========================

lr_raw_classweight_default = Pipeline(steps=[
    ("preprocess", preprocess_raw),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
        class_weight="balanced"   # only change vs baseline
    ))
])

# Fit on TRAIN
lr_raw_classweight_default.fit(X_train_raw, y_train)

# Predict on EVAL (NOT test)
y_pred_cw = lr_raw_classweight_default.predict(X_eval_raw)
y_proba_cw = lr_raw_classweight_default.predict_proba(X_eval_raw)

# Evaluate on EVAL
metrics_lr_cw = evaluate_binary_classifier(y_eval, y_pred_cw, y_proba_cw)
print(metrics_lr_cw)

# Diagnostics
print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_cw))
print("\nClassification report:\n", classification_report(y_eval, y_pred_cw, digits=3))

# Log experiment
log_experiment(
    exp_id="LR__raw__classweight__default",
    model_family="LR",
    features="raw",
    sampling="classweight",
    tuning="default",
    notes="LogReg with class_weight='balanced' to address class imbalance (evaluated on validation set)",
    metrics=metrics_lr_cw,
    params=lr_raw_classweight_default.get_params()
)

results_df()

In [ ]:
# ==========================
# Model 3: LR with Threshold Tuning (RAW FEATURES)
# LR__raw__classweight__thresh_tuned
# 
# Procedure:
#   - Use the same LR class_weight model (already fitted)
#   - Get predicted probabilities on eval
#   - Sweep thresholds (0.05 → 0.95)
#   - Choose threshold that maximizes MCC (or target sensitivity)
# ==========================

# Get probabilities from the balanced LR model (already fitted)
y_proba_cw_pos = y_proba_cw[:, 1]  # probability of class 1 (diabetic)

# Sweep thresholds and compute metrics
thresholds_to_try = np.arange(0.05, 0.96, 0.01)

threshold_results = []
for thresh in thresholds_to_try:
    y_pred_thresh = (y_proba_cw_pos >= thresh).astype(int)
    
    # Compute key metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_eval, y_pred_thresh, labels=[0, 1], zero_division=0
    )
    
    mcc = matthews_corrcoef(y_eval, y_pred_thresh)
    bal_acc = balanced_accuracy_score(y_eval, y_pred_thresh)
    f1_diabetic = f1[1]
    sensitivity = recall[1]  # recall for diabetic class
    specificity = recall[0]  # recall for non-diabetic class
    ppv = precision[1]       # precision for diabetic class
    
    threshold_results.append({
        "threshold": thresh,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "f1_diabetic": f1_diabetic,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
    })

df_thresh = pd.DataFrame(threshold_results)

# Find optimal thresholds for different objectives
best_mcc_idx = df_thresh["mcc"].idxmax()
best_f1_idx = df_thresh["f1_diabetic"].idxmax()

# Extract best threshold (maximize MCC)
best_threshold = df_thresh.loc[best_mcc_idx, "threshold"]

print(f"Best threshold (max MCC): {best_threshold:.3f}")
print(f"MCC at best threshold: {df_thresh.loc[best_mcc_idx, 'mcc']:.4f}")

# Target sensitivity threshold (e.g., sensitivity >= 0.80)
target_sensitivity = 0.80
df_high_sens = df_thresh[df_thresh["sensitivity"] >= target_sensitivity]
if len(df_high_sens) > 0:
    best_sens_target_idx = df_high_sens["mcc"].idxmax()
    best_sens_target_thresh = df_thresh.loc[best_sens_target_idx, "threshold"]
    print(f"Best threshold achieving sensitivity ≥ {target_sensitivity}: {best_sens_target_thresh:.3f}")
else:
    best_sens_target_thresh = None
    print(f"No threshold achieves sensitivity ≥ {target_sensitivity}")

# Apply best threshold and evaluate
y_pred_thresh_tuned = (y_proba_cw_pos >= best_threshold).astype(int)

metrics_lr_thresh = evaluate_binary_classifier(y_eval, y_pred_thresh_tuned, y_proba_cw)
print("\nMetrics with tuned threshold:")
print(metrics_lr_thresh)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_thresh_tuned))
print("\nClassification report:\n", classification_report(y_eval, y_pred_thresh_tuned, digits=3))

# Log experiment
log_experiment(
    exp_id="LR__raw__classweight__thresh_tuned",
    model_family="LR",
    features="raw",
    sampling="classweight",
    tuning="thresh_tuned",
    notes=f"LR class_weight='balanced' with threshold={best_threshold:.2f} (maximizes MCC on eval set)",
    metrics=metrics_lr_thresh,
    params={**lr_raw_classweight_default.get_params(), "threshold": best_threshold}
)

results_df()

**Feature engineering**
- Now that a model baseline has been established, its important to engineer features


In [ ]:
corr = X_train_raw.corr(numeric_only=True)
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap = 'viridis', annot=True, fmt=".2f")
plt.title("Correlation matrix (raw features) - training set")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout()
savefig("corr_matrix_raw_train.png")
plt.show()


In [ ]:
# List of highly correlated features, threshold = 0.7 (initial value)
# unable to fully identify from the graph

# Find highly correlated feature pairs (absolute corr >= threshold)
threshold = 0.4

corr_abs = corr.abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_corr"})
    .sort_values("abs_corr", ascending=False)
)

high_corr_pairs_filtered = high_corr_pairs[high_corr_pairs["abs_corr"] >= threshold]

print(f"Number of pairs with abs(corr) >= {threshold}: {len(high_corr_pairs_filtered)}")
high_corr_pairs_filtered.head(20)

**Create new features**

In [ ]:
# Create features
# 1 RiskFactorCount = sum of major risk flags - captures cumulative burden
# 2 BMI x physical activity interaction - BMI changes depending on activity
# 3 Age x high BP - BP risk increases with age
# 4 log1p mental health and physical health. they are skewed. 

# ===========================
# Create feature set A
# ===========================

X_train_fe_A = X_train_raw.copy()
X_test_fe_A = X_test_raw.copy()
X_eval_fe_A = X_eval_raw.copy()

# Add features
# 1 RiskFactorCount
risk_flags_A = ["HighBP","HighChol","Smoker","Stroke","HeartDiseaseorAttack","DiffWalk"]
X_train_fe_A["RiskFactorCount"] = X_train_fe_A[risk_flags_A].sum(axis=1)
X_test_fe_A["RiskFactorCount"] = X_test_fe_A[risk_flags_A].sum(axis=1)
X_eval_fe_A["RiskFactorCount"] = X_eval_fe_A[risk_flags_A].sum(axis=1)

# 2 BMI x physical activity interaction
X_train_fe_A["BMI_PhysActivity"] = X_train_fe_A["BMI"] * X_train_fe_A["PhysActivity"]
X_test_fe_A["BMI_PhysActivity"] = X_test_fe_A["BMI"] * X_test_fe_A["PhysActivity"]
X_eval_fe_A["BMI_PhysActivity"] = X_eval_fe_A["BMI"] * X_eval_fe_A["PhysActivity"]

# 3 Age x high BP interaction
X_train_fe_A["Age_HighBP"] = X_train_fe_A["Age"] * X_train_fe_A["HighBP"]
X_test_fe_A["Age_HighBP"] = X_test_fe_A["Age"] * X_test_fe_A["HighBP"]
X_eval_fe_A["Age_HighBP"] = X_eval_fe_A["Age"] * X_eval_fe_A["HighBP"]

# 4 log1p mental health and physical health (skewed)
X_train_fe_A["Log1p_MentHlth"] = np.log1p(X_train_fe_A["MentHlth"])
X_test_fe_A["Log1p_MentHlth"] = np.log1p(X_test_fe_A["MentHlth"])
X_eval_fe_A["Log1p_MentHlth"] = np.log1p(X_eval_fe_A["MentHlth"])
X_train_fe_A["Log1p_PhysHlth"] = np.log1p(X_train_fe_A["PhysHlth"])
X_test_fe_A["Log1p_PhysHlth"] = np.log1p(X_test_fe_A["PhysHlth"])
X_eval_fe_A["Log1p_PhysHlth"] = np.log1p(X_eval_fe_A["PhysHlth"])

# check everything worked as planned
print("Feature set A created")
print("Raw feature count: ", X_train_raw.shape[1])
print("Engineered feature count: ", X_train_fe_A.shape[1])

new_columns_A = ["RiskFactorCount", "BMI_PhysActivity", "Age_HighBP", "Log1p_MentHlth", "Log1p_PhysHlth"]

print ("New columns added in train:", all(c in X_train_fe_A.columns for c in new_columns_A))
print ("New columns added in test:", all(c in X_test_fe_A.columns for c in new_columns_A))
print ("New columns added in eval:", all(c in X_eval_fe_A.columns for c in new_columns_A))

In [ ]:
# Save feature set A to a single CSV (with target + split)
feature_set_A_path = PROJECT_ROOT / "data" / "processed" / "CDC_Diabetes_Dataset_feature_set_A.csv"

train_df = X_train_fe_A.copy()
train_df["Diabetes_binary"] = y_train.values
train_df["split"] = "train"

eval_df = X_eval_fe_A.copy()
eval_df["Diabetes_binary"] = y_eval.values
eval_df["split"] = "eval"

test_df = X_test_fe_A.copy()
test_df["Diabetes_binary"] = y_test.values
test_df["split"] = "test"

feature_set_A_all = pd.concat([train_df, eval_df, test_df], ignore_index=True)
feature_set_A_all.to_csv(feature_set_A_path, index=False)

print(f"Feature set A saved to {feature_set_A_path}")


In [ ]:
# ==========================
# LR on Feature Set A (no balancing)
# LR__featA__none__default
# ==========================

# 1) Update feature lists to include engineered numeric columns
engineered_numeric_A = ["RiskFactorCount", "BMI_PhysActivity", "Age_HighBP", "Log1p_MentHlth", "Log1p_PhysHlth"]
numeric_features_A = numeric_features + engineered_numeric_A

preprocess_featA = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features_A),
        ("ord", StandardScaler(), ordinal_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop"
)

lr_featA_none_default = Pipeline(steps=[
    ("preprocess", preprocess_featA),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
        class_weight=None
    ))
])

# 2) Fit on TRAIN
lr_featA_none_default.fit(X_train_fe_A, y_train)

# 3) Evaluate on EVAL
y_pred_featA = lr_featA_none_default.predict(X_eval_fe_A)
y_proba_featA = lr_featA_none_default.predict_proba(X_eval_fe_A)

metrics_lr_featA = evaluate_binary_classifier(y_eval, y_pred_featA, y_proba_featA)
print(metrics_lr_featA)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_featA))
print("\nClassification report:\n", classification_report(y_eval, y_pred_featA, digits=3))

log_experiment(
    exp_id="LR__featA__none__default",
    model_family="LR",
    features="featA",
    sampling="none",
    tuning="default",
    notes="LR on Feature Set A engineered features, no balancing (eval set)",
    metrics=metrics_lr_featA,
    params=lr_featA_none_default.get_params()
)

results_df()

In [ ]:
# ==========================
# LR on Feature Set A (class weighted)
# LR__featA__classweight__default
# ==========================

lr_featA_classweight_default = Pipeline(steps=[
    ("preprocess", preprocess_featA),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
        class_weight="balanced"
    ))
])

# Fit on TRAIN
lr_featA_classweight_default.fit(X_train_fe_A, y_train)

# Evaluate on EVAL
y_pred_featA_cw = lr_featA_classweight_default.predict(X_eval_fe_A)
y_proba_featA_cw = lr_featA_classweight_default.predict_proba(X_eval_fe_A)

metrics_lr_featA_cw = evaluate_binary_classifier(
    y_eval, y_pred_featA_cw, y_proba_featA_cw
)
print(metrics_lr_featA_cw)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_featA_cw))
print("\nClassification report:\n", classification_report(y_eval, y_pred_featA_cw, digits=3))

log_experiment(
    exp_id="LR__featA__classweight__default",
    model_family="LR",
    features="featA",
    sampling="classweight",
    tuning="default",
    notes="LR on Feature Set A with class_weight='balanced' (eval set)",
    metrics=metrics_lr_featA_cw,
    params=lr_featA_classweight_default.get_params()
)

results_df()

In [ ]:
# ==========================
# LR on Feature Set A (balanced + threshold tuning)
# LR__featA__classweight__thresh_tuned
# ==========================

# Get probabilities from the balanced LR model on Feature Set A (already fitted)
y_proba_featA_cw_pos = y_proba_featA_cw[:, 1]  # probability of class 1 (diabetic)

# Sweep thresholds and compute metrics
thresholds_to_try = np.arange(0.05, 0.96, 0.01)

threshold_results_featA = []
for thresh in thresholds_to_try:
    y_pred_thresh = (y_proba_featA_cw_pos >= thresh).astype(int)
    
    # Compute key metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_eval, y_pred_thresh, labels=[0, 1], zero_division=0
    )
    
    mcc = matthews_corrcoef(y_eval, y_pred_thresh)
    bal_acc = balanced_accuracy_score(y_eval, y_pred_thresh)
    f1_diabetic = f1[1]
    sensitivity = recall[1]  # recall for diabetic class
    specificity = recall[0]  # recall for non-diabetic class
    ppv = precision[1]       # precision for diabetic class
    
    threshold_results_featA.append({
        "threshold": thresh,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "f1_diabetic": f1_diabetic,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
    })

df_thresh_featA = pd.DataFrame(threshold_results_featA)

# Find optimal threshold (maximize MCC)
best_mcc_idx_featA = df_thresh_featA["mcc"].idxmax()
best_threshold_featA = df_thresh_featA.loc[best_mcc_idx_featA, "threshold"]

print(f"Best threshold for Feature Set A (max MCC): {best_threshold_featA:.3f}")
print(f"MCC at best threshold: {df_thresh_featA.loc[best_mcc_idx_featA, 'mcc']:.4f}")

# Apply best threshold and evaluate
y_pred_featA_thresh_tuned = (y_proba_featA_cw_pos >= best_threshold_featA).astype(int)

metrics_lr_featA_thresh = evaluate_binary_classifier(y_eval, y_pred_featA_thresh_tuned, y_proba_featA_cw)
print("\nMetrics with tuned threshold (Feature Set A):")
print(metrics_lr_featA_thresh)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_featA_thresh_tuned))
print("\nClassification report:\n", classification_report(y_eval, y_pred_featA_thresh_tuned, digits=3))

# Log experiment
log_experiment(
    exp_id="LR__featA__classweight__thresh_tuned",
    model_family="LR",
    features="featA",
    sampling="classweight",
    tuning="thresh_tuned",
    notes=f"LR on Feature Set A with class_weight='balanced' and threshold={best_threshold_featA:.2f} (maximizes MCC on eval set)",
    metrics=metrics_lr_featA_thresh,
    params={**lr_featA_classweight_default.get_params(), "threshold": best_threshold_featA}
)

results_df()

In [ ]:
# ==========================
# XGBoost Baseline (BINARY)
# XGB__raw__none__default
# ==========================

xgb_raw_none_default = Pipeline(steps=[
    ("preprocess", preprocess_raw),
    ("clf", XGBClassifier(
        objective="binary:logistic",  #  Binary classification
        eval_metric="logloss",         #  Binary log loss
        random_state=42,
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9
    ))
])

# Fit on TRAIN
xgb_raw_none_default.fit(X_train_raw, y_train)

# Evaluate on EVAL
y_pred_xgb = xgb_raw_none_default.predict(X_eval_raw)
y_proba_xgb = xgb_raw_none_default.predict_proba(X_eval_raw)

metrics_xgb = evaluate_binary_classifier(y_eval, y_pred_xgb, y_proba_xgb)  #  Binary eval
print(metrics_xgb)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb, digits=3))

log_experiment(
    exp_id="XGB__raw__none__default",
    model_family="XGB",
    features="raw",
    sampling="none",
    tuning="default",
    notes="Baseline XGBoost on raw features, no class weighting, threshold=0.50 (eval set)",
    metrics=metrics_xgb,
    params=xgb_raw_none_default.get_params()
)

results_df()

In [ ]:
# ==========================
# XGB with class-balanced sample weights (BINARY)
# XGB__raw__classweight__default
# ==========================

# Compute per-row weights from class frequencies in TRAIN set only
sample_w = compute_sample_weight(class_weight="balanced", y=y_train)

xgb_raw_classweight_default = Pipeline(steps=[
    ("preprocess", preprocess_raw),
    ("clf", XGBClassifier(
        objective="binary:logistic",  # Binary classification
        eval_metric="logloss",         #  Binary log loss
        random_state=42,
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9
    ))
])

# Fit with sample weights (pass to the classifier step)
xgb_raw_classweight_default.fit(
    X_train_raw, y_train,
    clf__sample_weight=sample_w
)

# Evaluate on EVAL
y_pred_xgb_cw = xgb_raw_classweight_default.predict(X_eval_raw)
y_proba_xgb_cw = xgb_raw_classweight_default.predict_proba(X_eval_raw)

metrics_xgb_cw = evaluate_binary_classifier(y_eval, y_pred_xgb_cw, y_proba_xgb_cw)
print(metrics_xgb_cw)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb_cw))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb_cw, digits=3))

log_experiment(
    exp_id="XGB__raw__classweight__default",
    model_family="XGB",
    features="raw",
    sampling="classweight",
    tuning="default",
    notes="XGBoost with class-balanced sample weights, threshold=0.50 (eval set)",
    metrics=metrics_xgb_cw,
    params=xgb_raw_classweight_default.get_params()
)

results_df()

# ==========================
# Report-ready confusion matrix (counts + row %), XGB raw + classweight
# ==========================
class_names = ["Non-Diabetic", "Diabetic/Prediabetic"]

# Confusion matrix (counts)
cm_xgb_cw = confusion_matrix(y_eval, y_pred_xgb_cw, labels=[0, 1])
cm_xgb_cw_df = pd.DataFrame(cm_xgb_cw, index=class_names, columns=class_names)

# Row-normalized percentages
cm_xgb_cw_row_pct = cm_xgb_cw_df.div(cm_xgb_cw_df.sum(axis=1), axis=0) * 100

# Combine counts + percentages for annotation
cm_xgb_cw_annot = cm_xgb_cw_df.astype(int).astype(str) + "\n(" + cm_xgb_cw_row_pct.round(1).astype(str) + "%)"

# Display table (counts + row %) for direct copy into report
cm_xgb_cw_table = cm_xgb_cw_df.copy()
for r in cm_xgb_cw_table.index:
    for c in cm_xgb_cw_table.columns:
        cm_xgb_cw_table.loc[r, c] = f"{cm_xgb_cw_df.loc[r, c]} ({cm_xgb_cw_row_pct.loc[r, c]:.1f}%)"

print("Confusion matrix (counts and row %):")
display(cm_xgb_cw_table)

# Plot heatmap for report-ready figure
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_xgb_cw_df,
    annot=cm_xgb_cw_annot,
    fmt="",
    cmap="Blues",
    cbar=False,
    linewidths=0.5,
    linecolor="white"
 )
plt.title("Confusion Matrix — XGB Class-Weighted (Eval Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
savefig("confusion_matrix_xgb_classweight_eval.png")
plt.show()

In [ ]:
# ==========================
# XGB with class-balanced sample weights + threshold tuning (BINARY)
# XGB__raw__classweight__thresh_tuned
# 
# Procedure:
#   - Use the fitted xgb_raw_classweight_default model
#   - Get predicted probabilities on eval
#   - Sweep thresholds (0.05 → 0.95)
#   - Choose threshold that maximizes MCC
# ==========================

# Get probabilities from the class-weighted XGB model (already fitted)
y_proba_xgb_cw_pos = y_proba_xgb_cw[:, 1]  # probability of class 1 (diabetic)

# Sweep thresholds and compute metrics
thresholds_to_try = np.arange(0.05, 0.96, 0.01)

threshold_results_xgb = []
for thresh in thresholds_to_try:
    y_pred_thresh = (y_proba_xgb_cw_pos >= thresh).astype(int)
    
    # Compute key metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_eval, y_pred_thresh, labels=[0, 1], zero_division=0
    )
    
    mcc = matthews_corrcoef(y_eval, y_pred_thresh)
    bal_acc = balanced_accuracy_score(y_eval, y_pred_thresh)
    f1_diabetic = f1[1]
    sensitivity = recall[1]  # recall for diabetic class
    specificity = recall[0]  # recall for non-diabetic class
    ppv = precision[1]       # precision for diabetic class
    
    threshold_results_xgb.append({
        "threshold": thresh,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "f1_diabetic": f1_diabetic,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
    })

df_thresh_xgb = pd.DataFrame(threshold_results_xgb)

# Find optimal threshold (maximize MCC)
best_mcc_idx_xgb = df_thresh_xgb["mcc"].idxmax()
best_threshold_xgb = df_thresh_xgb.loc[best_mcc_idx_xgb, "threshold"]

print(f"Best threshold for XGB (max MCC): {best_threshold_xgb:.3f}")
print(f"MCC at best threshold: {df_thresh_xgb.loc[best_mcc_idx_xgb, 'mcc']:.4f}")

# Apply best threshold and evaluate
y_pred_xgb_thresh_tuned = (y_proba_xgb_cw_pos >= best_threshold_xgb).astype(int)

metrics_xgb_thresh = evaluate_binary_classifier(y_eval, y_pred_xgb_thresh_tuned, y_proba_xgb_cw)
print("\nMetrics with tuned threshold (XGB):")
print(metrics_xgb_thresh)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb_thresh_tuned))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb_thresh_tuned, digits=3))

# Log experiment
log_experiment(
    exp_id="XGB__raw__classweight__thresh_tuned",
    model_family="XGB",
    features="raw",
    sampling="classweight",
    tuning="thresh_tuned",
    notes=f"XGBoost with class-balanced sample weights and threshold={best_threshold_xgb:.2f} (maximizes MCC on eval set)",
    metrics=metrics_xgb_thresh,
    params={**xgb_raw_classweight_default.get_params(), "threshold": best_threshold_xgb}
)

results_df()

In [ ]:
# ==========================
# XGB on Feature Set A with class-balanced sample weights (BINARY)
# XGB__featA__classweight__default
# ==========================

# Compute per-row weights from class frequencies in TRAIN set only
sample_w = compute_sample_weight(class_weight="balanced", y=y_train)

xgb_featA_classweight_default = Pipeline(steps=[
    ("preprocess", preprocess_featA),
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9
    ))
])

# Fit with sample weights (pass to the classifier step)
xgb_featA_classweight_default.fit(
    X_train_fe_A, y_train,
    clf__sample_weight=sample_w
)

# Evaluate on EVAL
y_pred_xgb_featA_cw = xgb_featA_classweight_default.predict(X_eval_fe_A)
y_proba_xgb_featA_cw = xgb_featA_classweight_default.predict_proba(X_eval_fe_A)

metrics_xgb_featA_cw = evaluate_binary_classifier(y_eval, y_pred_xgb_featA_cw, y_proba_xgb_featA_cw)
print(metrics_xgb_featA_cw)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb_featA_cw))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb_featA_cw, digits=3))

log_experiment(
    exp_id="XGB__featA__classweight__default",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning="default",
    notes="XGBoost on Feature Set A with class-balanced sample weights, threshold=0.50 (eval set)",
    metrics=metrics_xgb_featA_cw,
    params=xgb_featA_classweight_default.get_params()
)

results_df()

In [ ]:
# ==========================
# XGB on Feature Set A with class-balanced sample weights + threshold tuning (BINARY)
# XGB__featA__classweight__thresh_tuned
# 
# Procedure:
#   - Use the fitted xgb_featA_classweight_default model
#   - Get predicted probabilities on eval
#   - Sweep thresholds (0.05 → 0.95)
#   - Choose threshold that maximizes MCC
# ==========================

# Get probabilities from the class-weighted XGB model on Feature Set A (already fitted)
y_proba_xgb_featA_cw_pos = y_proba_xgb_featA_cw[:, 1]  # probability of class 1 (diabetic)

# Sweep thresholds and compute metrics
thresholds_to_try = np.arange(0.05, 0.96, 0.01)

threshold_results_xgb_featA = []
for thresh in thresholds_to_try:
    y_pred_thresh = (y_proba_xgb_featA_cw_pos >= thresh).astype(int)
    
    # Compute key metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_eval, y_pred_thresh, labels=[0, 1], zero_division=0
    )
    
    mcc = matthews_corrcoef(y_eval, y_pred_thresh)
    bal_acc = balanced_accuracy_score(y_eval, y_pred_thresh)
    f1_diabetic = f1[1]
    sensitivity = recall[1]  # recall for diabetic class
    specificity = recall[0]  # recall for non-diabetic class
    ppv = precision[1]       # precision for diabetic class
    
    threshold_results_xgb_featA.append({
        "threshold": thresh,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "f1_diabetic": f1_diabetic,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
    })

df_thresh_xgb_featA = pd.DataFrame(threshold_results_xgb_featA)

# Find optimal threshold (maximize MCC)
best_mcc_idx_xgb_featA = df_thresh_xgb_featA["mcc"].idxmax()
best_threshold_xgb_featA = df_thresh_xgb_featA.loc[best_mcc_idx_xgb_featA, "threshold"]

print(f"Best threshold for XGB Feature Set A (max MCC): {best_threshold_xgb_featA:.3f}")
print(f"MCC at best threshold: {df_thresh_xgb_featA.loc[best_mcc_idx_xgb_featA, 'mcc']:.4f}")

# Apply best threshold and evaluate
y_pred_xgb_featA_thresh_tuned = (y_proba_xgb_featA_cw_pos >= best_threshold_xgb_featA).astype(int)

metrics_xgb_featA_thresh = evaluate_binary_classifier(y_eval, y_pred_xgb_featA_thresh_tuned, y_proba_xgb_featA_cw)
print("\nMetrics with tuned threshold (XGB Feature Set A):")
print(metrics_xgb_featA_thresh)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb_featA_thresh_tuned))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb_featA_thresh_tuned, digits=3))

# Log experiment
log_experiment(
    exp_id="XGB__featA__classweight__thresh_tuned",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning="thresh_tuned",
    notes=f"XGBoost on Feature Set A with class-balanced sample weights and threshold={best_threshold_xgb_featA:.2f} (maximizes MCC on eval set)",
    metrics=metrics_xgb_featA_thresh,
    params={**xgb_featA_classweight_default.get_params(), "threshold": best_threshold_xgb_featA}
)

results_df()

In [ ]:
# ==========================
# XGB Optuna Hyperparameter Tuning (BINARY) on Feature Set A
# XGB__featA__classweight__optuna
# 
# Optimizes PR-AUC (Precision-Recall AUC) via 5-fold stratified CV on TRAIN set
# Uses class-balanced sample weights
# ==========================

# Safety checks
y_train_array = np.asarray(y_train).astype(int)
assert set(np.unique(y_train_array)).issubset({0, 1}), "y_train must be binary {0,1}"
print("Train positive rate (diabetic=1):", y_train_array.mean())

# Compute balanced sample weights for TRAIN set
sw_train = compute_sample_weight(class_weight="balanced", y=y_train_array)

# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    """
    Optuna objective function: tries different XGBoost hyperparameters
    and returns mean PR-AUC across 5 CV folds.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0),
    }

    clf = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        **params
    )

    pipe = Pipeline(steps=[
        ("preprocess", preprocess_featA),  # Use Feature Set A preprocessor
        ("clf", clf),
    ])

    # Cross-validation loop
    ap_scores = []
    for train_idx, val_idx in cv.split(X_train_fe_A, y_train_array):
        X_tr, X_va = X_train_fe_A.iloc[train_idx], X_train_fe_A.iloc[val_idx]
        y_tr, y_va = y_train_array[train_idx], y_train_array[val_idx]
        sw_tr = sw_train[train_idx]

        pipe.fit(X_tr, y_tr, clf__sample_weight=sw_tr)
        p_va = pipe.predict_proba(X_va)[:, 1]  # probability of diabetic class
        ap_scores.append(average_precision_score(y_va, p_va))

    return float(np.mean(ap_scores))

# Create Optuna study and optimize
print("Starting Optuna hyperparameter search (40 trials)...")
study = optuna.create_study(direction="maximize", study_name="XGB_featA_binary")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("\n" + "="*60)
print("OPTUNA OPTIMIZATION COMPLETE")
print("="*60)
print(f"Best CV PR-AUC: {study.best_value:.4f}")
print(f"Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Fit final model on FULL TRAIN set using best params
best_xgb_featA = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    **study.best_params
)

xgb_featA_optuna = Pipeline(steps=[
    ("preprocess", preprocess_featA),
    ("clf", best_xgb_featA),
])

xgb_featA_optuna.fit(X_train_fe_A, y_train, clf__sample_weight=sw_train)
print("✅ Trained xgb_featA_optuna on full TRAIN set")

# Evaluate on EVAL set
y_pred_xgb_optuna = xgb_featA_optuna.predict(X_eval_fe_A)
y_proba_xgb_optuna = xgb_featA_optuna.predict_proba(X_eval_fe_A)

metrics_xgb_optuna = evaluate_binary_classifier(y_eval, y_pred_xgb_optuna, y_proba_xgb_optuna)
print("\nEvaluation metrics on EVAL set:")
print(metrics_xgb_optuna)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb_optuna))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb_optuna, digits=3))

# Log experiment
log_experiment(
    exp_id="XGB__featA__classweight__optuna",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning="optuna",
    notes=f"XGBoost on Feature Set A with Optuna-tuned hyperparameters (PR-AUC={study.best_value:.4f}), evaluated on validation set",
    metrics=metrics_xgb_optuna,
    params={**study.best_params, "cv_pr_auc": study.best_value}
)

results_df()

In [ ]:
# ==========================
# XGB on Feature Set A with Optuna + Threshold Tuning (BINARY)
# XGB__featA__classweight__optuna_thresh_tuned
# 
# Procedure:
#   - Use the fitted xgb_featA_optuna model (Optuna-optimized)
#   - Get predicted probabilities on EVAL
#   - Sweep thresholds (0.05 → 0.95)
#   - Choose threshold that maximizes MCC
# ==========================

# Get probabilities from the Optuna-tuned XGB model (already fitted)
y_proba_xgb_optuna_pos = y_proba_xgb_optuna[:, 1]  # probability of class 1 (diabetic)

# Sweep thresholds and compute metrics
thresholds_to_try = np.arange(0.05, 0.96, 0.01)

threshold_results_xgb_optuna = []
for thresh in thresholds_to_try:
    y_pred_thresh = (y_proba_xgb_optuna_pos >= thresh).astype(int)
    
    # Compute key metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_eval, y_pred_thresh, labels=[0, 1], zero_division=0
    )
    
    mcc = matthews_corrcoef(y_eval, y_pred_thresh)
    bal_acc = balanced_accuracy_score(y_eval, y_pred_thresh)
    f1_diabetic = f1[1]
    sensitivity = recall[1]  # recall for diabetic class
    specificity = recall[0]  # recall for non-diabetic class
    ppv = precision[1]       # precision for diabetic class
    
    threshold_results_xgb_optuna.append({
        "threshold": thresh,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "f1_diabetic": f1_diabetic,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
    })

df_thresh_xgb_optuna = pd.DataFrame(threshold_results_xgb_optuna)

# Find optimal threshold (maximize MCC)
best_mcc_idx_optuna = df_thresh_xgb_optuna["mcc"].idxmax()
best_threshold_optuna = df_thresh_xgb_optuna.loc[best_mcc_idx_optuna, "threshold"]

print(f"Best threshold for Optuna-tuned XGB (max MCC): {best_threshold_optuna:.3f}")
print(f"MCC at best threshold: {df_thresh_xgb_optuna.loc[best_mcc_idx_optuna, 'mcc']:.4f}")

# Optional: Show threshold that maximizes F1-score for comparison
best_f1_idx = df_thresh_xgb_optuna["f1_diabetic"].idxmax()
best_f1_thresh = df_thresh_xgb_optuna.loc[best_f1_idx, "threshold"]
print(f"Threshold maximizing F1 (diabetic): {best_f1_thresh:.3f} (F1={df_thresh_xgb_optuna.loc[best_f1_idx, 'f1_diabetic']:.4f})")

# Apply best threshold and evaluate
y_pred_xgb_optuna_thresh_tuned = (y_proba_xgb_optuna_pos >= best_threshold_optuna).astype(int)

metrics_xgb_optuna_thresh = evaluate_binary_classifier(y_eval, y_pred_xgb_optuna_thresh_tuned, y_proba_xgb_optuna)
print("\nMetrics with tuned threshold (Optuna XGB):")
print(metrics_xgb_optuna_thresh)

print("\nConfusion matrix:\n", confusion_matrix(y_eval, y_pred_xgb_optuna_thresh_tuned))
print("\nClassification report:\n", classification_report(y_eval, y_pred_xgb_optuna_thresh_tuned, digits=3))

# Log experiment
log_experiment(
    exp_id="XGB__featA__classweight__optuna_thresh_tuned",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning="optuna+thresh",
    notes=f"Optuna-tuned XGBoost on Feature Set A with threshold={best_threshold_optuna:.2f} (maximizes MCC on eval set)",
    metrics=metrics_xgb_optuna_thresh,
    params={**study.best_params, "cv_pr_auc": study.best_value, "threshold": best_threshold_optuna}
)

results_df()

# ==========================
# Report-ready confusion matrix (counts + row %), XGB Optuna + threshold
# ==========================
class_names = ["Non-Diabetic", "Diabetic/Prediabetic"]

# Confusion matrix (counts)
cm_xgb_optuna = confusion_matrix(y_eval, y_pred_xgb_optuna_thresh_tuned, labels=[0, 1])
cm_xgb_optuna_df = pd.DataFrame(cm_xgb_optuna, index=class_names, columns=class_names)

# Row-normalized percentages
cm_xgb_optuna_row_pct = cm_xgb_optuna_df.div(cm_xgb_optuna_df.sum(axis=1), axis=0) * 100

# Combine counts + percentages for annotation
cm_xgb_optuna_annot = cm_xgb_optuna_df.astype(int).astype(str) + "\n(" + cm_xgb_optuna_row_pct.round(1).astype(str) + "%)"

# Display table (counts + row %) for direct copy into report
cm_xgb_optuna_table = cm_xgb_optuna_df.copy()
for r in cm_xgb_optuna_table.index:
    for c in cm_xgb_optuna_table.columns:
        cm_xgb_optuna_table.loc[r, c] = f"{cm_xgb_optuna_df.loc[r, c]} ({cm_xgb_optuna_row_pct.loc[r, c]:.1f}%)"

print("Confusion matrix (counts and row %):")
display(cm_xgb_optuna_table)

# Plot heatmap for report-ready figure
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_xgb_optuna_df,
    annot=cm_xgb_optuna_annot,
    fmt="",
    cmap="Blues",
    cbar=False,
    linewidths=0.5,
    linecolor="white"
 )
plt.title("Confusion Matrix — XGB Optuna + Threshold (Eval Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
savefig("confusion_matrix_xgb_optuna_thresh_eval.png")
plt.show()

**Calibration and final eval**

In [ ]:
# ==========================
# CALIBRATION WORKFLOW FOR BINARY CLASSIFICATION
# Selection criterion: highest Recall (sensitivity) for diabetic class
# ==========================

# Compare calibration methods on EVAL set
print("="*60)
print("CALIBRATION METHOD COMPARISON (selecting by Recall)")
print("="*60)

# Pre-calibration baseline
y_proba_eval_pre = xgb_featA_optuna.predict_proba(X_eval_fe_A)[:, 1]
brier_pre = brier_score_loss(y_eval, y_proba_eval_pre)
logloss_pre = log_loss(y_eval, xgb_featA_optuna.predict_proba(X_eval_fe_A))
recall_pre = recall_score(y_eval, (y_proba_eval_pre >= 0.5).astype(int), pos_label=1)

# Sigmoid calibration (Platt scaling)
cal_sigmoid = CalibratedClassifierCV(
    estimator=clone(xgb_featA_optuna),
    method="sigmoid",
    cv=3
)
cal_sigmoid.fit(X_train_fe_A, y_train)
y_proba_eval_sigmoid = cal_sigmoid.predict_proba(X_eval_fe_A)[:, 1]
brier_sigmoid = brier_score_loss(y_eval, y_proba_eval_sigmoid)
logloss_sigmoid = log_loss(y_eval, cal_sigmoid.predict_proba(X_eval_fe_A))
recall_sigmoid = recall_score(y_eval, (y_proba_eval_sigmoid >= 0.5).astype(int), pos_label=1)

# Isotonic calibration
cal_isotonic = CalibratedClassifierCV(
    estimator=clone(xgb_featA_optuna),
    method="isotonic",
    cv=3
)
cal_isotonic.fit(X_train_fe_A, y_train)
y_proba_eval_isotonic = cal_isotonic.predict_proba(X_eval_fe_A)[:, 1]
brier_isotonic = brier_score_loss(y_eval, y_proba_eval_isotonic)
logloss_isotonic = log_loss(y_eval, cal_isotonic.predict_proba(X_eval_fe_A))
recall_isotonic = recall_score(y_eval, (y_proba_eval_isotonic >= 0.5).astype(int), pos_label=1)

# Comparison table — sorted by Recall (descending), then Brier (ascending) as tiebreak
calibration_comparison = pd.DataFrame({
    "Method": ["Uncalibrated", "Sigmoid (Platt)", "Isotonic"],
    "Recall (sensitivity)": [recall_pre, recall_sigmoid, recall_isotonic],
    "Brier Score": [brier_pre, brier_sigmoid, brier_isotonic],
    "Log Loss": [logloss_pre, logloss_sigmoid, logloss_isotonic]
}).sort_values(["Recall (sensitivity)", "Brier Score"], ascending=[False, True])

print("\n📋 CALIBRATION COMPARISON (sorted by Recall):")
display(calibration_comparison)

# Select best method (highest Recall, tiebreak by lowest Brier)
best_method = calibration_comparison.iloc[0]["Method"]
print(f"\n✅ Best calibration method (by Recall): {best_method}")

method_to_model = {
    "Uncalibrated": xgb_featA_optuna,
    "Sigmoid (Platt)": cal_sigmoid,
    "Isotonic": cal_isotonic
}
best_calibrated_model = method_to_model[best_method]
chosen_method = best_method

print(f"Selected model: {chosen_method}")

# Re-tune threshold on CALIBRATED probabilities
y_proba_cal_eval = best_calibrated_model.predict_proba(X_eval_fe_A)[:, 1]

threshold_results_cal = []
for thresh in np.arange(0.05, 0.96, 0.01):
    y_pred_thresh = (y_proba_cal_eval >= thresh).astype(int)
    mcc = matthews_corrcoef(y_eval, y_pred_thresh)
    threshold_results_cal.append({"threshold": thresh, "mcc": mcc})

df_thresh_cal = pd.DataFrame(threshold_results_cal)
best_threshold_final = df_thresh_cal.loc[df_thresh_cal["mcc"].idxmax(), "threshold"]

print(f"\n✅ Final threshold (on {chosen_method} model): {best_threshold_final:.3f}")

# Assign final model
final_model = (best_calibrated_model, best_threshold_final)
print(f"Final model: {type(final_model[0]).__name__} with threshold={best_threshold_final:.4f}")

In [ ]:
# ==========================
# Calibration Curve (Before/After) — All 3 methods
# ==========================

# Bin probabilities into 10 quantiles
prob_true_uncal, prob_pred_uncal = calibration_curve(
    y_eval, y_proba_eval_pre, n_bins=10, strategy='quantile'
)
prob_true_sigmoid, prob_pred_sigmoid = calibration_curve(
    y_eval, y_proba_eval_sigmoid, n_bins=10, strategy='quantile'
)
prob_true_isotonic, prob_pred_isotonic = calibration_curve(
    y_eval, y_proba_eval_isotonic, n_bins=10, strategy='quantile'
)

plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect Calibration')
plt.plot(prob_pred_uncal, prob_true_uncal, 's-', lw=2, 
         label=f'Uncalibrated (Brier={brier_pre:.3f})', color='#d62728')
plt.plot(prob_pred_sigmoid, prob_true_sigmoid, '^-', lw=2, 
         label=f'Sigmoid (Brier={brier_sigmoid:.3f})', color='#1f77b4')
plt.plot(prob_pred_isotonic, prob_true_isotonic, 'o-', lw=2, 
         label=f'Isotonic (Brier={brier_isotonic:.3f})', color='#2ca02c')
plt.xlabel('Predicted Probability', fontsize=12)
plt.ylabel('True Fraction of Positives', fontsize=12)
plt.title(f'Probability Calibration Curve\n(Selected: {chosen_method} by Recall)', fontweight='bold')
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
savefig("calibration_curve_comparison.png")
plt.show()

print(f"✅ Calibration curve saved (selected method: {chosen_method})")

In [ ]:
# ==========================
# Final Model Evaluation on TEST Set
# ==========================

print("="*60)
print("FINAL MODEL EVALUATION — HELD-OUT TEST SET")
print("="*60)

# Unpack final model (using calibrated model from calibration cell)
model_pipeline, threshold = final_model

print(f"Using: {chosen_method}-calibrated XGBoost with threshold = {threshold:.3f}")
print("="*60)

# 1) Get predictions on TEST set
y_proba_test = model_pipeline.predict_proba(X_test_fe_A)[:, 1]
y_pred_test = (y_proba_test >= threshold).astype(int)

# 2) Compute comprehensive metrics
metrics_test = evaluate_binary_classifier(y_test, y_pred_test, model_pipeline.predict_proba(X_test_fe_A))

print("\n📊 TEST SET METRICS:")
print(f"   Brier Score: {metrics_test['brier_score']:.4f}")
print(f"   Log Loss: {metrics_test['log_loss']:.4f}")
print(f"   Accuracy: {metrics_test['accuracy']:.4f}")
print(f"   Balanced Accuracy: {metrics_test['balanced_accuracy']:.4f}")
print(f"   F1 (diabetic): {metrics_test['f1_class_1 (diabetic)']:.4f}")
print(f"   Sensitivity (recall): {metrics_test['recall_class_1 (sensitivity)']:.4f}")
print(f"   Specificity: {metrics_test['recall_class_0 (specificity)']:.4f}")
print(f"   PPV (precision): {metrics_test['precision_class_1 (PPV)']:.4f}")
print(f"   NPV: {metrics_test['precision_class_0 (NPV)']:.4f}")
print(f"   MCC: {metrics_test['mcc']:.4f}")
print(f"   ROC-AUC: {metrics_test['roc_auc']:.4f}")
print(f"   PR-AUC: {metrics_test['pr_auc']:.4f}")

# 3) Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
print("\n📋 CONFUSION MATRIX (TEST SET):")
print(cm)

# Calculate per-class counts for annotation
tn, fp, fn, tp = cm.ravel()
print(f"\nBreakdown:")
print(f"  True Negatives (TN):  {tn:,}")
print(f"  False Positives (FP): {fp:,}")
print(f"  False Negatives (FN): {fn:,}")
print(f"  True Positives (TP):  {tp:,}")

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Diabetic', 'Diabetic/Prediabetic'],
            yticklabels=['Non-Diabetic', 'Diabetic/Prediabetic'],
            cbar_kws={'label': 'Count'})
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title(f'Confusion Matrix — Final Model - Calibrated XGBoost)\nTest Set', 
          fontweight='bold', fontsize=13)
savefig("confusion_matrix_test.png")
plt.show()

# 4) Classification Report
print("\n📝 CLASSIFICATION REPORT (TEST SET):")
print(classification_report(y_test, y_pred_test, 
                           target_names=['Non-Diabetic', 'Diabetic/Prediabetic'], 
                           digits=4))

# 5) Determine calibration method label for logging
cal_method_label = chosen_method.lower().replace(" (platt)", "").replace(" ", "_")

# 6) Log final test metrics to experiment tracker
log_experiment(
    exp_id=f"XGB__featA__classweight__optuna_{cal_method_label}_final",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning=f"optuna+{cal_method_label}+thresh",
    notes=f"FINAL TEST SET: {chosen_method}-calibrated XGBoost with threshold={threshold:.3f}",
    metrics=metrics_test,
    params={
        **study.best_params,
        "calibration": cal_method_label,
        "threshold": threshold,
        "cv_pr_auc": study.best_value
    }
)

print("\n✅ Final model evaluation complete")
print(f"✅ Test set size: {len(y_test):,} samples")
print(f"✅ Model: {type(model_pipeline).__name__} ({chosen_method}) with threshold={threshold:.4f}")

In [ ]:
# ==========================
# Final Model Evaluation — UNCALIBRATED Optuna XGBoost (Best Recall)
# ==========================

print("="*60)
print("FINAL MODEL — UNCALIBRATED OPTUNA XGBOOST (BEST RECALL)")
print("="*60)

# Use the uncalibrated Optuna-tuned XGBoost pipeline
uncal_model = xgb_featA_optuna

# Re-tune threshold on uncalibrated probabilities (EVAL set)
y_proba_uncal_eval = uncal_model.predict_proba(X_eval_fe_A)[:, 1]

thresh_results_uncal = []
for t in np.arange(0.05, 0.96, 0.01):
    yp = (y_proba_uncal_eval >= t).astype(int)
    thresh_results_uncal.append({
        "threshold": t,
        "mcc": matthews_corrcoef(y_eval, yp)
    })
df_t_uncal = pd.DataFrame(thresh_results_uncal)
thresh_uncal = df_t_uncal.loc[df_t_uncal["mcc"].idxmax(), "threshold"]

print(f"Optimal threshold (max MCC on eval): {thresh_uncal:.3f}")

# ---------- TEST SET ----------
y_proba_test_uncal = uncal_model.predict_proba(X_test_fe_A)[:, 1]
y_pred_test_uncal = (y_proba_test_uncal >= thresh_uncal).astype(int)

metrics_uncal = evaluate_binary_classifier(
    y_test, y_pred_test_uncal, uncal_model.predict_proba(X_test_fe_A)
)

print("\n📊 TEST SET METRICS (Uncalibrated):")
print(f"   Brier Score:          {metrics_uncal['brier_score']:.4f}")
print(f"   Log Loss:             {metrics_uncal['log_loss']:.4f}")
print(f"   Accuracy:             {metrics_uncal['accuracy']:.4f}")
print(f"   Balanced Accuracy:    {metrics_uncal['balanced_accuracy']:.4f}")
print(f"   F1 (diabetic):        {metrics_uncal['f1_class_1 (diabetic)']:.4f}")
print(f"   Sensitivity (recall): {metrics_uncal['recall_class_1 (sensitivity)']:.4f}")
print(f"   Specificity:          {metrics_uncal['recall_class_0 (specificity)']:.4f}")
print(f"   PPV (precision):      {metrics_uncal['precision_class_1 (PPV)']:.4f}")
print(f"   NPV:                  {metrics_uncal['precision_class_0 (NPV)']:.4f}")
print(f"   MCC:                  {metrics_uncal['mcc']:.4f}")
print(f"   ROC-AUC:              {metrics_uncal['roc_auc']:.4f}")
print(f"   PR-AUC:               {metrics_uncal['pr_auc']:.4f}")

# Confusion matrix with counts + row %
cm_uncal = confusion_matrix(y_test, y_pred_test_uncal)
tn, fp, fn, tp = cm_uncal.ravel()

class_names = ["Non-Diabetic", "Diabetic/Prediabetic"]
cm_uncal_df = pd.DataFrame(cm_uncal, index=class_names, columns=class_names)
cm_uncal_pct = cm_uncal_df.div(cm_uncal_df.sum(axis=1), axis=0) * 100

cm_uncal_annot = cm_uncal_df.copy().astype(str)
for r in class_names:
    for c in class_names:
        cm_uncal_annot.loc[r, c] = f"{cm_uncal_df.loc[r, c]:,}\n({cm_uncal_pct.loc[r, c]:.1f}%)"

cm_uncal_table = cm_uncal_df.copy().astype(str)
for r in class_names:
    for c in class_names:
        cm_uncal_table.loc[r, c] = f"{cm_uncal_df.loc[r, c]:,} ({cm_uncal_pct.loc[r, c]:.1f}%)"

print("\n📋 CONFUSION MATRIX (counts + row %):")
display(cm_uncal_table)

print(f"\n  TN: {tn:,}  |  FP: {fp:,}")
print(f"  FN: {fn:,}  |  TP: {tp:,}")

plt.figure(figsize=(8, 6))
sns.heatmap(cm_uncal_df, annot=cm_uncal_annot, fmt='', cmap='Blues',
            cbar=False, linewidths=0.5, linecolor='white',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title(f'Confusion Matrix — Uncalibrated Optuna XGBoost\nTest Set (threshold={thresh_uncal:.3f})',
          fontweight='bold', fontsize=13)
savefig("confusion_matrix_test_uncalibrated.png")
plt.show()

print("\n📝 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_test_uncal,
                           target_names=class_names, digits=4))

log_experiment(
    exp_id="XGB__featA__classweight__optuna_uncalibrated_final",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning="optuna+uncalibrated+thresh",
    notes=f"FINAL TEST: Uncalibrated Optuna XGBoost, threshold={thresh_uncal:.3f}",
    metrics=metrics_uncal,
    params={**study.best_params, "calibration": "uncalibrated", "threshold": thresh_uncal}
)

print(f"\n✅ Uncalibrated model evaluation complete (threshold={thresh_uncal:.3f})")

In [ ]:
# ==========================
# Final Model Evaluation — CALIBRATED Optuna XGBoost
# Compares Sigmoid vs Isotonic, picks the one with better Brier Score
# ==========================

print("="*60)
print("FINAL MODEL — CALIBRATED OPTUNA XGBOOST")
print("="*60)

# --- Evaluate BOTH calibrated models on EVAL to pick the best ---
cal_candidates = {
    "Sigmoid (Platt)": cal_sigmoid,
    "Isotonic": cal_isotonic
}

cal_eval_results = []
for name, model in cal_candidates.items():
    p1 = model.predict_proba(X_eval_fe_A)[:, 1]
    brier = brier_score_loss(y_eval, p1)
    ll = log_loss(y_eval, model.predict_proba(X_eval_fe_A))
    
    # Find best threshold by MCC
    best_mcc, best_t = -1, 0.5
    for t in np.arange(0.05, 0.96, 0.01):
        m = matthews_corrcoef(y_eval, (p1 >= t).astype(int))
        if m > best_mcc:
            best_mcc, best_t = m, t
    
    recall_at_thresh = recall_score(y_eval, (p1 >= best_t).astype(int), pos_label=1)
    
    cal_eval_results.append({
        "Method": name, "Brier": brier, "Log Loss": ll,
        "Best MCC": best_mcc, "Threshold": best_t, "Recall": recall_at_thresh
    })

df_cal_eval = pd.DataFrame(cal_eval_results).sort_values("Brier", ascending=True)
print("\n📋 Calibrated model comparison (EVAL set):")
display(df_cal_eval)

# Pick the calibrated model with the lowest Brier score
best_cal_name = df_cal_eval.iloc[0]["Method"]
best_cal_thresh = df_cal_eval.iloc[0]["Threshold"]
best_cal_model = cal_candidates[best_cal_name]

print(f"\n✅ Selected calibrated model: {best_cal_name}")
print(f"✅ Threshold (max MCC on eval): {best_cal_thresh:.3f}")

# ---------- TEST SET ----------
y_proba_test_cal = best_cal_model.predict_proba(X_test_fe_A)[:, 1]
y_pred_test_cal = (y_proba_test_cal >= best_cal_thresh).astype(int)

metrics_cal = evaluate_binary_classifier(
    y_test, y_pred_test_cal, best_cal_model.predict_proba(X_test_fe_A)
)

print(f"\n📊 TEST SET METRICS ({best_cal_name} Calibrated):")
print(f"   Brier Score:          {metrics_cal['brier_score']:.4f}")
print(f"   Log Loss:             {metrics_cal['log_loss']:.4f}")
print(f"   Accuracy:             {metrics_cal['accuracy']:.4f}")
print(f"   Balanced Accuracy:    {metrics_cal['balanced_accuracy']:.4f}")
print(f"   F1 (diabetic):        {metrics_cal['f1_class_1 (diabetic)']:.4f}")
print(f"   Sensitivity (recall): {metrics_cal['recall_class_1 (sensitivity)']:.4f}")
print(f"   Specificity:          {metrics_cal['recall_class_0 (specificity)']:.4f}")
print(f"   PPV (precision):      {metrics_cal['precision_class_1 (PPV)']:.4f}")
print(f"   NPV:                  {metrics_cal['precision_class_0 (NPV)']:.4f}")
print(f"   MCC:                  {metrics_cal['mcc']:.4f}")
print(f"   ROC-AUC:              {metrics_cal['roc_auc']:.4f}")
print(f"   PR-AUC:               {metrics_cal['pr_auc']:.4f}")

# Confusion matrix with counts + row %
cm_cal = confusion_matrix(y_test, y_pred_test_cal)
tn, fp, fn, tp = cm_cal.ravel()

class_names = ["Non-Diabetic", "Diabetic/Prediabetic"]
cm_cal_df = pd.DataFrame(cm_cal, index=class_names, columns=class_names)
cm_cal_pct = cm_cal_df.div(cm_cal_df.sum(axis=1), axis=0) * 100

cm_cal_annot = cm_cal_df.copy().astype(str)
for r in class_names:
    for c in class_names:
        cm_cal_annot.loc[r, c] = f"{cm_cal_df.loc[r, c]:,}\n({cm_cal_pct.loc[r, c]:.1f}%)"

cm_cal_table = cm_cal_df.copy().astype(str)
for r in class_names:
    for c in class_names:
        cm_cal_table.loc[r, c] = f"{cm_cal_df.loc[r, c]:,} ({cm_cal_pct.loc[r, c]:.1f}%)"

print("\n📋 CONFUSION MATRIX (counts + row %):")
display(cm_cal_table)

print(f"\n  TN: {tn:,}  |  FP: {fp:,}")
print(f"  FN: {fn:,}  |  TP: {tp:,}")

cal_label_short = best_cal_name.split(" ")[0].lower()  # "sigmoid" or "isotonic"

plt.figure(figsize=(8, 6))
sns.heatmap(cm_cal_df, annot=cm_cal_annot, fmt='', cmap='Blues',
            cbar=False, linewidths=0.5, linecolor='white',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title(f'Confusion Matrix — {best_cal_name} Calibrated Optuna XGBoost\nTest Set (threshold={best_cal_thresh:.3f})',
          fontweight='bold', fontsize=13)
savefig(f"confusion_matrix_test_{cal_label_short}_calibrated.png")
plt.show()

print("\n📝 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_test_cal,
                           target_names=class_names, digits=4))

log_experiment(
    exp_id=f"XGB__featA__classweight__optuna_{cal_label_short}_final",
    model_family="XGB",
    features="featA",
    sampling="classweight",
    tuning=f"optuna+{cal_label_short}+thresh",
    notes=f"FINAL TEST: {best_cal_name} calibrated Optuna XGBoost, threshold={best_cal_thresh:.3f}",
    metrics=metrics_cal,
    params={**study.best_params, "calibration": cal_label_short, "threshold": best_cal_thresh}
)

# --- Set final_model for downstream cells (SHAP, LIME, etc.) ---
# Pick whichever has better MCC on test
if metrics_cal['mcc'] >= metrics_uncal['mcc']:
    final_model = (best_cal_model, best_cal_thresh)
    chosen_method = best_cal_name
    print(f"\n🏆 FINAL MODEL: {best_cal_name} Calibrated (better MCC on test)")
else:
    final_model = (uncal_model, thresh_uncal)
    chosen_method = "Uncalibrated"
    print(f"\n🏆 FINAL MODEL: Uncalibrated (better MCC on test)")

print(f"✅ final_model set for downstream cells (SHAP, LIME, permutation importance)")
print(f"✅ Model: {type(final_model[0]).__name__}, threshold={final_model[1]:.4f}")

In [ ]:
# ==========================
# ROC and Precision-Recall Curves (TEST SET)
# ==========================

# Use the ACTUAL final model (consistent with SHAP/LIME/permutation importance)
model_pipeline_final, threshold_final = final_model
y_proba_test_final = model_pipeline_final.predict_proba(X_test_fe_A)[:, 1]

# 1) ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba_test_final)
roc_auc_test = auc(fpr, tpr)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='#2ca02c', lw=2.5, 
         label=f'ROC Curve (AUC = {roc_auc_test:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title(f'ROC Curve — {chosen_method} XGBoost (Test Set)', fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# 2) Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba_test_final)
pr_auc_test = auc(recall_curve, precision_curve)

plt.subplot(1, 2, 2)
plt.plot(recall_curve, precision_curve, color='#ff7f0e', lw=2.5,
         label=f'PR Curve (AUC = {pr_auc_test:.3f})')
plt.axhline(y=y_test.mean(), color='k', linestyle='--', lw=2, 
            label=f'Baseline (prevalence = {y_test.mean():.3f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall (Sensitivity)', fontsize=12)
plt.ylabel('Precision (PPV)', fontsize=12)
plt.title(f'Precision-Recall Curve — {chosen_method} XGBoost (Test Set)', fontweight='bold')
plt.legend(loc="lower left")
plt.grid(alpha=0.3)

plt.tight_layout()
savefig("roc_pr_curves_test.png")
plt.show()

print(f"✅ Test set ROC-AUC: {roc_auc_test:.4f}")
print(f"✅ Test set PR-AUC: {pr_auc_test:.4f}")
print(f"✅ Model used: {chosen_method} (threshold={threshold_final:.3f})")

In [ ]:
# MCC vs Threshold plot — using the ACTUAL final model
model_pipeline_final, threshold_final = final_model
y_proba_final_eval = model_pipeline_final.predict_proba(X_eval_fe_A)[:, 1]

# Recompute MCC sweep on eval set for the final model
threshold_results_final = []
for thresh in np.arange(0.05, 0.96, 0.01):
    y_pred_t = (y_proba_final_eval >= thresh).astype(int)
    mcc = matthews_corrcoef(y_eval, y_pred_t)
    threshold_results_final.append({"threshold": thresh, "mcc": mcc})

df_thresh_final = pd.DataFrame(threshold_results_final)
thresholds = df_thresh_final["threshold"].values
mcc_vals = df_thresh_final["mcc"].values

# Find MCC at the optimal threshold
mcc_best = df_thresh_final.loc[
    (df_thresh_final["threshold"] - threshold_final).abs().idxmin(), "mcc"
]

plt.figure(figsize=(8, 6))
plt.plot(thresholds, mcc_vals, marker="o")
plt.axvline(threshold_final, linestyle="--")
plt.annotate(
    f"best = {threshold_final:.3f}\nMCC = {mcc_best:.3f}",
    xy=(threshold_final, mcc_best),
    xytext=(threshold_final + 0.05, mcc_best - 0.02),
    textcoords="data",
    size=14,
    arrowprops=dict(arrowstyle="->", color="black")
)

plt.xlabel("Threshold")
plt.ylabel("MCC")
plt.title(f"Matthews Correlation Coefficient vs Threshold\n({chosen_method} Model)", y=1.02)
plt.tight_layout()
savefig("mcc_vs_threshold_final.png")
plt.show()

print(f"✅ MCC plot uses: {chosen_method} (threshold={threshold_final:.3f}, MCC={mcc_best:.3f})")

**Explainability**

**Permutation Importance and XGB gain + comparison**

In [ ]:
# ==========================
# Permutation Feature Importance (BINARY)
# Goal:
#   - Compute permutation importance on EVAL set
#   - Use the FINAL CALIBRATED model
#   - Score using multiple metrics (MCC, F1, ROC-AUC)
#   - Create visualizations comparing to XGBoost built-in importance
# ==========================

from sklearn.inspection import permutation_importance

print("="*60)
print("PERMUTATION FEATURE IMPORTANCE")
print("="*60)

# 1) Use the FINAL calibrated model (isotonic-calibrated XGBoost)
model_pipeline, threshold = final_model

# 2) Compute permutation importance on EVAL set
# We'll use multiple scoring metrics to get a comprehensive view
print("\n⏳ Computing permutation importance (this may take a few minutes)...")

# MCC (Matthews Correlation Coefficient) - robust for imbalanced data
perm_importance_mcc = permutation_importance(
    model_pipeline, 
    X_eval_fe_A, 
    y_eval,
    scoring='matthews_corrcoef',
    n_repeats=30,  # Repeat shuffling 30 times for stability
    random_state=42,
    n_jobs=-1
)

# ROC-AUC (probability-based metric)
perm_importance_roc = permutation_importance(
    model_pipeline, 
    X_eval_fe_A, 
    y_eval,
    scoring='roc_auc',
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

# F1-score (diabetic class)
from sklearn.metrics import make_scorer, f1_score
f1_scorer = make_scorer(f1_score, pos_label=1)

perm_importance_f1 = permutation_importance(
    model_pipeline, 
    X_eval_fe_A, 
    y_eval,
    scoring=f1_scorer,
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

print("✅ Permutation importance computed")

# 3) Get feature names from the pipeline
# These are the ORIGINAL feature names (before preprocessing)
feature_names_orig = X_eval_fe_A.columns.tolist()

# 4) Create comprehensive results dataframe
perm_results = pd.DataFrame({
    'feature': feature_names_orig,
    'importance_mcc_mean': perm_importance_mcc.importances_mean,
    'importance_mcc_std': perm_importance_mcc.importances_std,
    'importance_roc_mean': perm_importance_roc.importances_mean,
    'importance_roc_std': perm_importance_roc.importances_std,
    'importance_f1_mean': perm_importance_f1.importances_mean,
    'importance_f1_std': perm_importance_f1.importances_std,
}).sort_values('importance_mcc_mean', ascending=False)

# 5) Display top features
print("\n" + "="*60)
print("TOP 15 FEATURES (Permutation Importance - MCC)")
print("="*60)
print(perm_results.head(15).to_string(index=False))

# 6) Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total features: {len(feature_names_orig)}")
print(f"Top feature (MCC): {perm_results.iloc[0]['feature']}")
print(f"Top importance (MCC): {perm_results.iloc[0]['importance_mcc_mean']:.4f} ± {perm_results.iloc[0]['importance_mcc_std']:.4f}")
print(f"Mean importance (MCC): {perm_results['importance_mcc_mean'].mean():.4f}")
print(f"Features with importance > 0.001: {(perm_results['importance_mcc_mean'] > 0.001).sum()}")

# 7) Check engineered features
engineered_feats = ["RiskFactorCount", "BMI_PhysActivity", "Age_HighBP", "Log1p_MentHlth", "Log1p_PhysHlth"]
top10_feats = set(perm_results.head(10)["feature"])
engineered_in_top10 = [f for f in engineered_feats if f in top10_feats]

print(f"\n✅ Engineered features in top 10: {len(engineered_in_top10)}")
if engineered_in_top10:
    print(f"   Features: {engineered_in_top10}")

# 8) Plot permutation importance (MCC)
plt.figure(figsize=(10, 8))
top_features = perm_results.head(20)
colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, 20))

plt.barh(
    range(len(top_features)),
    top_features['importance_mcc_mean'].values[::-1],
    xerr=top_features['importance_mcc_std'].values[::-1],
    color=colors[::-1],
    capsize=3
)
plt.yticks(range(len(top_features)), top_features['feature'].values[::-1])
plt.xlabel('Permutation Importance (MCC decrease)', fontsize=12)
plt.title('Permutation Feature Importance (MCC) — Diabetic/Prediabetic Risk\nFinal Calibrated Model', 
          fontweight='bold', fontsize=13)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
savefig("permutation_importance_mcc_binary.png")
plt.show()

# 9) Plot comparison across metrics (top 10 features)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
top10 = perm_results.head(10)

for ax, metric, title in zip(
    axes, 
    ['importance_mcc_mean', 'importance_roc_mean', 'importance_f1_mean'],
    ['MCC Decrease', 'ROC-AUC Decrease', 'F1 Decrease']
):
    ax.barh(range(10), top10[metric].values[::-1], color='steelblue')
    ax.set_yticks(range(10))
    ax.set_yticklabels(top10['feature'].values[::-1])
    ax.set_xlabel(f'Importance ({title})', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Permutation Importance — Comparison Across Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
savefig("permutation_importance_comparison_binary.png")
plt.show()

print("\n✅ Permutation importance analysis complete")

In [ ]:
# ==========================
# XGBoost Built-in Feature Importance (BINARY)
# Goal:
#   - Extract gain, weight, and cover importance from the trained XGBoost model
#   - Compare with permutation importance
#   - Visualize top features across different importance types
# ==========================

print("="*60)
print("XGBOOST BUILT-IN FEATURE IMPORTANCE")
print("="*60)

# 1) Extract the trained XGBoost classifier from the pipeline
xgb_clf = xgb_featA_optuna.named_steps["clf"]

# 2) Get feature names (after preprocessing)
feature_names_transformed = preprocess_featA.get_feature_names_out()

# 3) Extract different importance types
# Gain: average gain (improvement in accuracy) brought by a feature to the splits
importance_gain = xgb_clf.get_booster().get_score(importance_type='gain')

# Weight: number of times a feature appears in a tree
importance_weight = xgb_clf.get_booster().get_score(importance_type='weight')

# Cover: average coverage (number of samples) affected by splits on the feature
importance_cover = xgb_clf.get_booster().get_score(importance_type='cover')

# 4) Convert to dataframes (XGBoost uses f0, f1, f2... internally)
def importance_to_df(importance_dict, feature_names, importance_name):
    """Convert XGBoost importance dict to dataframe with proper feature names."""
    # Create mapping from f0, f1, ... to actual feature names
    feature_map = {f"f{i}": name for i, name in enumerate(feature_names)}
    
    # Convert to dataframe
    df = pd.DataFrame([
        {"feature": feature_map.get(k, k), importance_name: v}
        for k, v in importance_dict.items()
    ]).sort_values(importance_name, ascending=False).reset_index(drop=True)
    
    return df

df_gain = importance_to_df(importance_gain, feature_names_transformed, "gain")
df_weight = importance_to_df(importance_weight, feature_names_transformed, "weight")
df_cover = importance_to_df(importance_cover, feature_names_transformed, "cover")

# 5) Merge all importance types into one dataframe
xgb_importance = df_gain.merge(df_weight, on="feature", how="outer").merge(df_cover, on="feature", how="outer")
xgb_importance = xgb_importance.fillna(0).sort_values("gain", ascending=False)

# 6) Display top features
print("\n" + "="*60)
print("TOP 15 FEATURES (XGBoost Built-in Importance)")
print("="*60)
print(xgb_importance.head(15).to_string(index=False))

# 7) Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total features: {len(xgb_importance)}")
print(f"Top feature (gain): {xgb_importance.iloc[0]['feature']}")
print(f"Top gain: {xgb_importance.iloc[0]['gain']:.2f}")
print(f"Features used in trees: {(xgb_importance['weight'] > 0).sum()}")

# 8) Check engineered features
engineered_feats = ["RiskFactorCount", "BMI_PhysActivity", "Age_HighBP", "Log1p_MentHlth", "Log1p_PhysHlth"]
top10_xgb = set(xgb_importance.head(10)["feature"])
engineered_in_top10_xgb = [f for f in engineered_feats if any(f in feat for feat in top10_xgb)]

print(f"\n✅ Engineered features in top 10 (XGB gain): {len(engineered_in_top10_xgb)}")
if engineered_in_top10_xgb:
    print(f"   Features: {engineered_in_top10_xgb}")

# 9) Plot XGBoost importance (Gain)
plt.figure(figsize=(10, 8))
top_features_xgb = xgb_importance.head(20)
colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, 20))

plt.barh(
    range(len(top_features_xgb)),
    top_features_xgb['gain'].values[::-1],
    color=colors[::-1]
)
plt.yticks(range(len(top_features_xgb)), top_features_xgb['feature'].values[::-1])
plt.xlabel('XGBoost Importance (Gain)', fontsize=12)
plt.title('XGBoost Built-in Feature Importance (Gain) — Diabetic/Prediabetic Risk\nOptuna-tuned Model', 
          fontweight='bold', fontsize=13)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
savefig("xgboost_importance_gain_binary.png")
plt.show()

# 10) Plot comparison across importance types (top 10 features by gain)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
top10_xgb = xgb_importance.head(10)

for ax, metric, title in zip(
    axes, 
    ['gain', 'weight', 'cover'],
    ['Gain (Split Improvement)', 'Weight (Split Count)', 'Cover (Sample Count)']
):
    # Normalize for better visualization
    values = top10_xgb[metric].values
    if values.max() > 0:
        values = values / values.max()  # Normalize to [0, 1]
    
    ax.barh(range(10), values[::-1], color='coral')
    ax.set_yticks(range(10))
    ax.set_yticklabels(top10_xgb['feature'].values[::-1])
    ax.set_xlabel(f'Normalized {title}', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('XGBoost Built-in Importance — Comparison Across Types', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
savefig("xgboost_importance_comparison_binary.png")
plt.show()

print("\n✅ XGBoost built-in importance analysis complete")

In [ ]:
# ==========================
# COMPARISON: Permutation vs XGBoost Importance
# ==========================

print("="*60)
print("FEATURE IMPORTANCE COMPARISON")
print("="*60)

# Merge permutation and XGBoost importance
comparison = perm_results[['feature', 'importance_mcc_mean']].merge(
    xgb_importance[['feature', 'gain']], 
    on='feature', 
    how='outer'
).fillna(0)

# Normalize for comparison
comparison['perm_norm'] = comparison['importance_mcc_mean'] / comparison['importance_mcc_mean'].max()
comparison['xgb_norm'] = comparison['gain'] / comparison['gain'].max()

# Correlation between methods
corr = comparison[['perm_norm', 'xgb_norm']].corr().iloc[0, 1]
print(f"\n📊 Correlation between permutation and XGBoost importance: {corr:.3f}")

# Top 10 comparison
print("\nTop 10 by Permutation Importance:")
print(comparison.sort_values('importance_mcc_mean', ascending=False).head(10)[['feature', 'importance_mcc_mean', 'gain']])

print("\nTop 10 by XGBoost Gain:")
print(comparison.sort_values('gain', ascending=False).head(10)[['feature', 'gain', 'importance_mcc_mean']])

# Scatter plot comparison
plt.figure(figsize=(10, 8))
plt.scatter(comparison['xgb_norm'], comparison['perm_norm'], alpha=0.6, s=80)

# Label top features
top_features_comp = comparison.nlargest(10, 'perm_norm')
for _, row in top_features_comp.iterrows():
    plt.annotate(row['feature'], (row['xgb_norm'], row['perm_norm']), 
                fontsize=9, alpha=0.7)

plt.xlabel('XGBoost Importance (Gain, normalized)', fontsize=12)
plt.ylabel('Permutation Importance (MCC, normalized)', fontsize=12)
plt.title(f'Feature Importance Comparison\nCorrelation: {corr:.3f}', 
         fontweight='bold', fontsize=13)
plt.grid(alpha=0.3)
plt.tight_layout()
savefig("importance_comparison_scatter.png")
plt.show()

print("\n✅ Comparison complete")

**Global SHAP**

In [ ]:
# ==========================
# SHAP — Cell 1 (Setup) — BINARY CLASSIFICATION
# Goal:
#   - Use your OPTUNA-TUNED XGB pipeline (xgb_featA_optuna)
#   - Build a SHAP TreeExplainer on the *trained XGB model*
#   - Prepare transformed background data + feature names
# ==========================

# ✅ Use the correct variable name from your notebook
assert "preprocess" in xgb_featA_optuna.named_steps, "xgb_featA_optuna must be a fitted Pipeline with a 'preprocess' step."
assert "clf" in xgb_featA_optuna.named_steps, "xgb_featA_optuna must be a fitted Pipeline with a 'clf' step."

# Extract preprocessor + fitted classifier
pre = xgb_featA_optuna.named_steps["preprocess"]
clf = xgb_featA_optuna.named_steps["clf"]

# Background sample (keep small-ish for speed)
# ✅ Use X_train_fe_A (Feature Set A with engineered features)
bg = X_train_fe_A.sample(2000, random_state=42)
X_bg = pre.transform(bg)

# Feature names after preprocessing (matches X_bg columns)
feature_names = pre.get_feature_names_out()

# Build explainer (TreeExplainer is the right choice for XGBoost tree models)
explainer = shap.TreeExplainer(
    clf,
    data=X_bg,
    feature_names=feature_names
)

print("✅ SHAP Cell 1 complete")
print("Background shape:", X_bg.shape)
print("Transformed feature count:", len(feature_names))
print(f"Model type: {type(clf).__name__}")
print(f"Objective: {clf.objective}")  # Should show 'binary:logistic'

In [ ]:
# ==========================
# SHAP — Cell 2 (Compute SHAP on a sample of EVAL rows) — BINARY
# Goal:
#   - Sample 5000 rows from X_eval_fe_A (Feature Set A)
#   - Transform them with the SAME preprocessor
#   - Compute SHAP values
#   - Handle binary classification output (no class dimension OR 2D array)
#   - Create mapping from eval *position* -> shap row
# ==========================

# 1) Sample eval ROW POSITIONS (0..len(eval)-1), NOT global dataframe indices
rng = np.random.default_rng(42)
eval_pos = rng.choice(len(X_eval_fe_A), size=5000, replace=False)

# 2) Get raw rows from Feature Set A, then transform using preprocessor
X_eval_sample_raw = X_eval_fe_A.iloc[eval_pos]
X_shap = pre.transform(X_eval_sample_raw)  # numpy array (n_samples, n_features)

# 3) Compute SHAP values
shap_values = explainer.shap_values(X_shap)

# 4) Handle binary classification output
# For binary classification, XGBoost SHAP returns either:
#   - A single array (n_samples, n_features) for class 1 (diabetic)
#   - OR a list [class0_shap, class1_shap]
print(f"SHAP output type: {type(shap_values)}")

if isinstance(shap_values, list):
    # If list, extract class 1 (diabetic/prediabetic) explanations
    print(f"SHAP returned list with {len(shap_values)} classes")
    shap_values_class1 = shap_values[1]  # Class 1 = Diabetic/Prediabetic
else:
    # Single array: already class 1 explanations
    print("SHAP returned single array (class 1 explanations)")
    shap_values_class1 = shap_values

# 5) Verify shape
print(f"Final SHAP shape: {shap_values_class1.shape}")
assert shap_values_class1.shape == (len(eval_pos), len(feature_names)), \
    f"Expected ({len(eval_pos)}, {len(feature_names)}), got {shap_values_class1.shape}"

# 6) Create mapping: eval position -> row within shap array
pos_to_shap = {int(p): i for i, p in enumerate(eval_pos)}

print("\n✅ SHAP Cell 2 complete (BINARY)")
print(f"X_shap shape: {X_shap.shape}")
print(f"SHAP array shape: {shap_values_class1.shape}")
print(f"Example mapping: eval pos {int(eval_pos[0])} -> shap row {pos_to_shap[int(eval_pos[0])]}")
print(f"Feature count: {shap_values_class1.shape[1]}")

In [ ]:
# ==========================
# SHAP — Cell 3 (Global Feature Importance) — BINARY
# Goal:
#   - Compute mean(|SHAP|) for the DIABETIC class (class 1)
#   - Create bar chart showing top features driving diabetes risk
#   - Generate summary table
# ==========================

# For binary classification, we only have class 1 (diabetic) SHAP values
# No class dimension — shap_values_class1 shape is (n_samples, n_features)

# 1) Compute mean absolute SHAP value per feature
mean_abs_shap = np.mean(np.abs(shap_values_class1), axis=0)

# 2) Create importance dataframe
imp_global = (
    pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

# 3) Plot top N features
topN = 15

plt.figure(figsize=(10, 7))
colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, topN))
plt.barh(
    imp_global.loc[:topN-1, "feature"][::-1], 
    imp_global.loc[:topN-1, "mean_abs_shap"][::-1],
    color=colors[::-1]
)
plt.xlabel("Mean(|SHAP value|)", fontsize=13)
plt.title("Global Feature Importance (SHAP) — Diabetic/Prediabetic Risk\nBinary Classification", 
          fontweight='bold', fontsize=14)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
savefig("shap_global_importance_binary.png")
plt.show()

# 4) Display top 15 features
print("="*60)
print("TOP 15 FEATURES DRIVING DIABETES RISK")
print("="*60)
print(imp_global.head(15).to_string(index=True))

# 5) Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total features: {len(feature_names)}")
print(f"Top feature: {imp_global.iloc[0]['feature']}")
print(f"Top feature importance: {imp_global.iloc[0]['mean_abs_shap']:.4f}")
print(f"Mean importance (all features): {mean_abs_shap.mean():.4f}")
print(f"Std importance: {mean_abs_shap.std():.4f}")

# 6) Check if engineered features appear in top 10
engineered_feats = ["RiskFactorCount", "BMI_PhysActivity", "Age_HighBP", "Log1p_MentHlth", "Log1p_PhysHlth"]
top10_feats = set(imp_global.head(10)["feature"])
engineered_in_top10 = [f for f in engineered_feats if any(f in feat for feat in top10_feats)]

print(f"\n✅ Engineered features in top 10: {len(engineered_in_top10)}")
if engineered_in_top10:
    print(f"   Features: {engineered_in_top10}")

print("\n✅ SHAP Cell 3 complete (BINARY)")

In [ ]:
# ==========================
# SHAP — Cell 4 (Beeswarm Summary Plot) — BINARY
# Goal:
#   - Create SHAP summary plot showing:
#     • Feature importance (vertical axis)
#     • Distribution of SHAP values (horizontal spread)
#     • Feature values (color: red=high, blue=low)
# ==========================

# Create SHAP Explanation object for summary plot
shap_explanation = shap.Explanation(
    values=shap_values_class1,
    base_values=np.full(len(shap_values_class1), explainer.expected_value),
    data=X_shap,
    feature_names=feature_names
)

# 1) Full summary plot (all features)
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_explanation,
    max_display=20,  # Show top 20 features
    show=False
)
plt.title("SHAP Summary Plot — Diabetic/Prediabetic Risk (Binary Classification)", 
          fontweight='bold', fontsize=13, pad=15)
plt.xlabel("SHAP value (impact on model output)", fontsize=11)
plt.tight_layout()
savefig("shap_summary_beeswarm_binary.png")
plt.show()

# 2) Bar plot version (cleaner for reports)
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_explanation,
    plot_type="bar",
    max_display=15,
    show=False
)
plt.title("SHAP Feature Importance — Diabetic/Prediabetic Risk (Bar Plot)", 
          fontweight='bold', fontsize=13)
plt.xlabel("Mean |SHAP value|", fontsize=11)
plt.tight_layout()
savefig("shap_summary_bar_binary.png")
plt.show()

print("\n✅ SHAP Cell 4 complete (BINARY)")
print("📊 Created 2 plots:")
print("   1. Beeswarm plot (shows feature value distributions)")
print("   2. Bar plot (cleaner for reports)")

**Local SHAP**

In [ ]:
# ==========================
# Local SHAP — 8 Case Types (Binary)
# Uses calibrated probs for selection, SHAP from TreeExplainer
# ==========================

print("="*60)
print("LOCAL SHAP — 8 CASES (BINARY)")
print("="*60)

# 1) Use calibrated model + threshold for predictions
model_pipeline, threshold = final_model
p1_eval = model_pipeline.predict_proba(X_eval_fe_A)[:, 1]
y_pred_eval = (p1_eval >= threshold).astype(int)

# 2) Build evaluation table (by eval POSITION)
df_eval = pd.DataFrame({
    "eval_pos": np.arange(len(y_eval)),
    "y_true": np.asarray(y_eval).astype(int),
    "y_pred": y_pred_eval,
    "p1": p1_eval
})

# 3) Keep only rows that exist in SHAP sample (from Cell 2)
df_eval = df_eval[df_eval["eval_pos"].isin(pos_to_shap.keys())].copy()
print("Rows available for local SHAP (in SHAP sample):", len(df_eval))

# 4) Helper to pick rows
def pick_row(df, desc, ascending):
    if df.empty:
        print(f"⚠️ No candidates for {desc}")
        return None
    return int(df.sort_values("p1", ascending=ascending)["eval_pos"].iloc[0])

# 5) Define 8 cases
cases = {
    "TP_high_conf":  pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==1)], "TP_high_conf", ascending=False),
    "TP_borderline":pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==1)], "TP_borderline", ascending=True),
    "FN_worst":     pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==0)], "FN_worst", ascending=True),
    "FN_border":    pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==0)], "FN_border", ascending=False),
    "FP_high_conf": pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==1)], "FP_high_conf", ascending=False),
    "FP_border":    pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==1)], "FP_border", ascending=True),
    "TN_high_conf": pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==0)], "TN_high_conf", ascending=True),
    "TN_border":    pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==0)], "TN_border", ascending=False),
}

print("Chosen eval positions:", cases)

# 6) Plot local SHAP (waterfall) for class 1
def plot_local_shap(eval_pos, tag, max_display=12):
    if eval_pos is None:
        return
    shap_row = pos_to_shap[eval_pos]
    sv = shap_values_class1[shap_row, :]
    x_row = X_shap[shap_row, :]
    r = df_eval[df_eval.eval_pos == eval_pos].iloc[0]

    base_val = explainer.expected_value
    exp = shap.Explanation(
        values=sv,
        base_values=base_val,
        data=x_row,
        feature_names=feature_names
    )

    title = f"{tag} | true={int(r.y_true)} pred={int(r.y_pred)} | p1={r.p1:.3f}"
    shap.plots.waterfall(exp, max_display=max_display, show=False)
    plt.title(title)
    savefig(f"shap_local_{tag}.png")
    plt.show()

for tag, pos in cases.items():
    plot_local_shap(pos, tag)

print("✅ Local SHAP (binary) complete")

**LIME**

In [ ]:
# ==========================
# Local LIME — 8 Case Types (Binary)
# ==========================
print("="*60)
print("LOCAL LIME — 8 CASES (BINARY)")
print("="*60)

# 1) Use calibrated model + threshold for predictions
model_pipeline, threshold = final_model
p1_eval = model_pipeline.predict_proba(X_eval_fe_A)[:, 1]
y_pred_eval = (p1_eval >= threshold).astype(int)

# 2) Build evaluation table (by eval POSITION)
df_eval = pd.DataFrame({
    "eval_pos": np.arange(len(y_eval)),
    "y_true": np.asarray(y_eval).astype(int),
    "y_pred": y_pred_eval,
    "p1": p1_eval
})

# 3) Helper to pick rows
def pick_row(df, desc, ascending):
    if df.empty:
        print(f"⚠️ No candidates for {desc}")
        return None
    return int(df.sort_values("p1", ascending=ascending)["eval_pos"].iloc[0])

# 4) Define 8 cases (same logic as SHAP)
cases = {
    "TP_high_conf":  pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==1)], "TP_high_conf", ascending=False),
    "TP_borderline":pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==1)], "TP_borderline", ascending=True),
    "FN_worst":     pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==0)], "FN_worst", ascending=True),
    "FN_border":    pick_row(df_eval[(df_eval.y_true==1) & (df_eval.y_pred==0)], "FN_border", ascending=False),
    "FP_high_conf": pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==1)], "FP_high_conf", ascending=False),
    "FP_border":    pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==1)], "FP_border", ascending=True),
    "TN_high_conf": pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==0)], "TN_high_conf", ascending=True),
    "TN_border":    pick_row(df_eval[(df_eval.y_true==0) & (df_eval.y_pred==0)], "TN_border", ascending=False),
}

print("Chosen eval positions:", cases)

# 5) LIME explainer setup
feature_names_lime = X_train_fe_A.columns.tolist()

categorical_cols = set(binary_features + ordinal_features)
categorical_indices = [i for i, c in enumerate(feature_names_lime) if c in categorical_cols]

lime_explainer = LimeTabularExplainer(
    training_data=X_train_fe_A.values,
    feature_names=feature_names_lime,
    class_names=["Non-Diabetic", "Diabetic/Prediabetic"],
    categorical_features=categorical_indices,
    discretize_continuous=True,
    mode="classification",
    random_state=42
)

# 6) Wrapper so LIME passes a DataFrame (ColumnTransformer needs column names)
def lime_predict_proba(X_np):
    X_df = pd.DataFrame(X_np, columns=feature_names_lime)
    return model_pipeline.predict_proba(X_df)

# 7) Plot local LIME explanations
def plot_local_lime(eval_pos, tag, num_features=10):
    if eval_pos is None:
        return
    x_row = X_eval_fe_A.iloc[eval_pos].values
    r = df_eval[df_eval.eval_pos == eval_pos].iloc[0]

    exp = lime_explainer.explain_instance(
        x_row,
        lime_predict_proba,
        num_features=num_features
    )

    fig = exp.as_pyplot_figure()
    title = f"{tag} | true={int(r.y_true)} pred={int(r.y_pred)} | p1={r.p1:.3f}"
    plt.title(title)
    savefig(f"lime_local_{tag}.png")
    plt.show()


for tag, pos in cases.items():
    plot_local_lime(pos, tag)

print("✅ Local LIME complete")

In [ ]:
# ==========================
# Side-by-side Local SHAP vs Local LIME (per case)
# ==========================
print("="*60)
print("SIDE-BY-SIDE LOCAL SHAP vs LIME")
print("="*60)

# Use the SAME cases chosen in the previous Local SHAP/LIME cells
case_positions = [p for p in cases.values() if p is not None]
if len(case_positions) == 0:
    raise ValueError("No valid case positions found in `cases`.")

# Compute SHAP ONLY for those case rows (so it matches previous plots)
X_eval_case_raw = X_eval_fe_A.iloc[case_positions]
X_shap = pre.transform(X_eval_case_raw)
shap_values_case = explainer.shap_values(X_shap)

if isinstance(shap_values_case, list):
    shap_values_class1 = shap_values_case[1]
else:
    shap_values_class1 = shap_values_case

# Map eval_pos -> row in this small SHAP batch
pos_to_shap = {int(p): i for i, p in enumerate(case_positions)}

def _plot_local_shap_bar(ax, eval_pos, max_display=10):
    shap_row = pos_to_shap[eval_pos]
    sv = shap_values_class1[shap_row, :]
    x_row = X_shap[shap_row, :]
    feat_vals = dict(zip(feature_names, x_row))

    idx = np.argsort(np.abs(sv))[-max_display:]
    feats = [feature_names[i] for i in idx]
    vals = sv[idx]

    labels = [f"{f} = {feat_vals[f]:.3f}" for f in feats]

    ax.barh(range(len(feats)), vals, color=["#d62728" if v > 0 else "#1f77b4" for v in vals])
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color="k", lw=1)
    ax.set_title("Local SHAP (bar)", fontsize=11)
    ax.set_xlabel("SHAP value")

def _plot_local_lime_bar(ax, exp, max_display=10):
    items = exp.as_list()[:max_display]
    feats = [f for f, _ in items][::-1]
    vals = [v for _, v in items][::-1]

    ax.barh(range(len(feats)), vals, color=["#d62728" if v > 0 else "#1f77b4" for v in vals])
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats, fontsize=8)
    ax.axvline(0, color="k", lw=1)
    ax.set_title("Local LIME (bar)", fontsize=11)
    ax.set_xlabel("LIME weight")

def plot_local_pair(eval_pos, tag, num_features=10):
    if eval_pos is None:
        return
    if eval_pos not in pos_to_shap:
        print(f"⚠️ Skipping {tag}: eval_pos {eval_pos} not in SHAP batch")
        return

    r = df_eval[df_eval.eval_pos == eval_pos].iloc[0]
    x_row = X_eval_fe_A.iloc[eval_pos].values

    exp_lime = lime_explainer.explain_instance(
        x_row,
        lime_predict_proba,
        num_features=num_features
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    _plot_local_shap_bar(axes[0], eval_pos, max_display=num_features)
    _plot_local_lime_bar(axes[1], exp_lime, max_display=num_features)

    title = f"{tag} | true={int(r.y_true)} pred={int(r.y_pred)} | p1={r.p1:.3f}"
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    savefig(f"shap_lime_side_by_side_{tag}.png")
    plt.show()

for tag, pos in cases.items():
    plot_local_pair(pos, tag, num_features=10)

print("✅ Side-by-side plots saved")